# Chapitre 5 — Classer et prendre une décision · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch05_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre : tant qu'une réponse manque, les vérifications le signalent sans erreur. Le corrigé se trouve dans `ch05_corrige.ipynb`, sur la même page compagnon.

**Sommaire.** Données · Les calculs des leçons 5.1 à 5.5 · Exercices 5.1 à 5.6

**Conventions du notebook.**
- Règle de décision (équation 5.1) : un message est signalé si sa probabilité estimée est **supérieure ou égale** au seuil. Pour les douze messages, la règle s'applique à la valeur **non arrondie** `p_exact`, recalculée par la sigmoïde ; le tableau du livre n'en montre que l'arrondi au centième. Dans les exercices où les probabilités sont données en décimales, ces décimales sont les valeurs à utiliser.
- Une mesure dont le dénominateur est nul (par exemple la précision quand aucune alerte n'est émise) vaut `None` et s'affiche « indéfinie ».
- Le F1 est calculé par $2\,VP / (2\,VP + FP + FN)$ (équation 5.8).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : une couleur = un rôle
SPAM, LEGITIME, SIGNALE, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
SEUIL, ORANGE, BLEU, VERT = '#8e24aa', '#eb6834', '#2a78d6', '#2e8b57'   # seuil ; sigmoïde ; perte ; filtre B


def sigmoide(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def affiche(v):
    """Convention du notebook : une mesure dont le dénominateur est nul vaut None et s'affiche « indéfinie »."""
    return 'indéfinie' if v is None else f'{v:.2f}'

## Données

Les identifiants D5.1, D5.2, D5.E1, D5.E2 et D5.E3 sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 5, recopiées des fichiers CSV du livre (ne pas modifier)

# D5.1 — douze messages : étiquette réelle, liens, mots suspects, probabilité estimée ARRONDIE au centième
douze = pd.DataFrame([
    ('M1', 'spam', 5, 4, 0.95),
    ('M2', 'legitime', 2, 0, 0.12),
    ('M3', 'legitime', 4, 0, 0.50),
    ('M4', 'spam', 3, 3, 0.62),
    ('M5', 'legitime', 1, 0, 0.05),
    ('M6', 'legitime', 2, 2, 0.27),
    ('M7', 'spam', 2, 3, 0.38),
    ('M8', 'legitime', 2, 1, 0.18),
    ('M9', 'spam', 4, 4, 0.88),
    ('M10', 'legitime', 1, 1, 0.08),
    ('M11', 'legitime', 0, 0, 0.02),
    ('M12', 'legitime', 0, 1, 0.03)
], columns=['message', 'etiquette', 'liens', 'mots_suspects', 'score'])

# D5.2 — filtre A : 1 000 messages par tranche (effectif, spams, probabilité moyenne prédite)
calibration = pd.DataFrame([
    (0.0, 0.1, 540, 22, 0.03),
    (0.1, 0.3, 150, 29, 0.19),
    (0.3, 0.5, 80, 33, 0.40),
    (0.5, 0.7, 60, 35, 0.60),
    (0.7, 0.9, 90, 71, 0.80),
    (0.9, 1.0, 80, 77, 0.95)
], columns=['tranche_min', 'tranche_max', 'messages', 'spams', 'proba_moyenne'])

# D5.E1 — filtre B : même tableau (exercice 5.1)
calibration_b = pd.DataFrame([
    (0.0, 0.1, 400, 40, 0.02),
    (0.1, 0.3, 175, 42, 0.18),
    (0.3, 0.5, 100, 43, 0.41),
    (0.5, 0.7, 100, 55, 0.61),
    (0.7, 0.9, 150, 93, 0.82),
    (0.9, 1.0, 75, 56, 0.97)
], columns=['tranche_min', 'tranche_max', 'messages', 'spams', 'proba_moyenne'])

# D5.E2 — dix messages de l'association (exercice 5.2) : ces décimales sont les valeurs à utiliser
dix = pd.DataFrame([
    ('N1', 'spam', 0.94),
    ('N2', 'legitime', 0.07),
    ('N3', 'legitime', 0.66),
    ('N4', 'spam', 0.50),
    ('N5', 'legitime', 0.30),
    ('N6', 'spam', 0.21),
    ('N7', 'legitime', 0.42),
    ('N8', 'spam', 0.77),
    ('N9', 'legitime', 0.13),
    ('N10', 'spam', 0.58)
], columns=['message', 'etiquette', 'score'])

# D5.E3 — quatre messages notés par deux filtres (exercice 5.3)
quatre = pd.DataFrame([
    ('P1', 'spam', 0.80, 0.99),
    ('P2', 'spam', 0.60, 0.90),
    ('P3', 'legitime', 0.40, 0.10),
    ('P4', 'legitime', 0.20, 0.01)
], columns=['message', 'etiquette', 'p_filtre_A', 'p_filtre_B'])

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 5.1 à 5.5.

In [ ]:
# Valeurs exactes des douze messages (équations 5.3 et 5.4) : c'est à elles que s'applique la règle
w, b = np.array([1, 0.5]), -4
X = douze[['liens', 'mots_suspects']].to_numpy()
z = X @ w + b
p_exact = sigmoide(z)
est_spam = (douze['etiquette'] == 'spam').to_numpy()
y = est_spam.astype(int)                 # 1 = spam, 0 = légitime

# Leçon 5.1 — déplacer le seuil (équations 5.1 et 5.2)
for seuil in (0.5, 0.25):
    signale = p_exact >= seuil                    # règle : supérieur OU ÉGAL, sur la valeur exacte
    assert (signale == (douze['score'] >= seuil)).all()   # ici, l'arrondi ne change aucune décision
    vp, fn = int((signale & est_spam).sum()), int((~signale & est_spam).sum())
    fp, vn = int((signale & ~est_spam).sum()), int((~signale & ~est_spam).sum())
    print(f'seuil {seuil} : VP = {vp}, FN = {fn}, FP = {fp}, VN = {vn} | '
          f'rappel = {vp}/{vp + fn} = {vp / (vp + fn):.3f}, '            # dénominateur : la ligne des spams
          f'fausses alertes = {fp}/{fp + vn} = {fp / (fp + vn):.3f}, '   # dénominateur : la ligne des légitimes
          f'précision = {vp}/{vp + fp} = {vp / (vp + fp):.3f}')         # dénominateur : la colonne des signalés
bascule = douze['message'][(p_exact >= 0.25) & (p_exact < 0.5)]
print('messages qui basculent de 0,5 à 0,25 :', list(bascule))

fig, axes = plt.subplots(2, 1, figsize=(7, 3.4), sharex=True)
for ax, seuil in zip(axes, (0.5, 0.25)):
    for q, s in zip(p_exact, est_spam):
        sig = q >= seuil
        ax.scatter(q, 1 if s else 0, s=70, marker='s' if s else 'o', color=SPAM if s else LEGITIME,
                   edgecolor=SIGNALE if sig else 'gray', linewidth=2.4 if sig else 0.6)
    ax.axvline(seuil, color=SEUIL, linewidth=2.5)
    ax.set_yticks([0, 1], ['légitimes', 'spams']); ax.set_ylim(-0.6, 1.6)
    ax.set_title(f'seuil {seuil}', loc='left', color=SEUIL)
axes[1].set_xlabel('Probabilité estimée (valeur exacte)'); plt.tight_layout(); plt.show()

In [ ]:
# Leçon 5.2 — score brut, probabilité estimée, décision ; le cas de M3 ; calibration
tableau = pd.DataFrame({'message': douze['message'], 'z': z, 'p_exact': p_exact.round(4),
                        'p arrondie': p_exact.round(2),
                        'décision (seuil 0,5)': np.where(p_exact >= 0.5, 'signalé', 'non')})
print(tableau.to_string(index=False))
print('arrondis = scores de la leçon 5.1 :', np.allclose(p_exact.round(2), douze['score']))
print(f"M3 : z = {z[2]}, e^0 = {np.exp(0.0)}, p̂ = 1/(1 + 1) = {p_exact[2]} exactement → "
      f"{'signalé' if p_exact[2] >= 0.5 else 'non signalé'} au seuil 0,5")
print(f'M4 : e^(-0,5) = {np.exp(-0.5):.4f}, sigmoïde(0,5) = {sigmoide(0.5):.3f} ; M7 : sigmoïde(-0,5) = {sigmoide(-0.5):.3f}')
print('seuil 0,5 sur p̂ ⇔ seuil 0 sur z :', bool(((p_exact >= 0.5) == (z >= 0)).all()),
      f'| seuil 0,25 sur p̂ ⇔ z ≥ {np.log(0.25 / 0.75):.2f}')

zs = np.linspace(-5, 4.5, 300)
plt.figure(figsize=(6, 3))
plt.plot(zs, sigmoide(zs), color=ORANGE, linewidth=2.2)
plt.axhline(0.5, color=SEUIL, linewidth=2)
plt.scatter(z[est_spam], p_exact[est_spam], marker='s', color=SPAM, zorder=3, label='spams')
plt.scatter(z[~est_spam], p_exact[~est_spam], marker='o', color=LEGITIME, edgecolor='gray', zorder=3, label='légitimes')
plt.xlabel('Score brut z'); plt.ylabel('Probabilité estimée p̂'); plt.legend(); plt.grid(alpha=0.4); plt.show()

# Une probabilité estimée se vérifie : côte à côte, p̂ moyenne prédite, fréquence observée, effectif (dénominateur)
cal = calibration.copy()
cal['fréquence observée'] = (cal['spams'] / cal['messages']).round(2)
print(cal[['tranche_min', 'tranche_max', 'proba_moyenne', 'fréquence observée', 'messages', 'spams']].to_string(index=False))
t = cal.iloc[4]
print(f"tranche 0,7–0,9 : prédit {t['proba_moyenne']:.2f}, observé {int(t['spams'])} / {int(t['messages'])} ≈ "
      f"{t['spams'] / t['messages']:.2f} | attendu {t['proba_moyenne'] * t['messages']:.0f} spams, "
      f"écart-type {np.sqrt(t['messages'] * 0.8 * 0.2):.1f}")
print('écart maximal entre prédit et observé :', f"{np.max(np.abs(cal['proba_moyenne'] - cal['spams'] / cal['messages'])):.3f}")

In [ ]:
# Leçon 5.3 — la pénalité d'un vrai spam : −ln(p̂) (équation 5.5). np.log est le logarithme NATUREL.
for p in (0.90, 0.50, 0.10, 0.01):
    print(f'p̂ = {p:.2f} → pénalité = {-np.log(p):.3f}')
print(f'rapports : {np.log(0.01) / np.log(0.5):.1f} fois la version à 0,50, '
      f'{np.log(0.01) / np.log(0.9):.1f} fois la version à 0,90')
p_spams = p_exact[est_spam]                             # valeurs NON arrondies de la leçon 5.2 (M4 : 0,622)
penalites = -np.log(p_spams)
print('spams', list(douze.loc[est_spam, 'message']), ': pénalités', penalites.round(2), f'| moyenne = {penalites.mean():.2f}')
print(f'produit des probabilités = {p_spams.prod():.3f} ; −ln(produit) = {-np.log(p_spams.prod()):.2f} '
      f'= somme des pénalités = {penalites.sum():.2f}')
ps = np.linspace(0.006, 1, 300)
plt.figure(figsize=(6, 3))
plt.plot(ps, -np.log(ps), color=BLEU, linewidth=2.2)
plt.scatter([0.9, 0.5, 0.01], -np.log([0.9, 0.5, 0.01]), marker='s', color=SPAM, zorder=3)
plt.xlabel('Probabilité estimée p̂ pour un vrai spam'); plt.ylabel('Pénalité −ln(p̂)'); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Leçon 5.4 — la perte de chaque message (équation 5.6) et la perte moyenne (équation 5.7)
def perte(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))   # −ln de la probabilité donnée à la bonne réponse

p_tab = douze['score'].to_numpy()                       # valeurs affichées dans le tableau (au centième)
pertes = perte(y, p_exact)                              # pertes calculées sur les valeurs NON arrondies, comme dans le livre
print(pd.DataFrame({'message': douze['message'], 'y': y, 'p̂': p_tab,
                    'bonne réponse': np.where(y == 1, p_tab, 1 - p_tab), 'perte': pertes.round(2)}).to_string(index=False))
print(f'M4 : −ln({p_exact[3]:.3f}) = {pertes[3]:.3f} ≈ {pertes[3]:.2f} (avec 0,62 arrondi, on trouverait {-np.log(0.62):.2f})')
print(f'M3 : −ln(1 − 0,50) = {-np.log(0.5):.3f} ; M6 : −ln(1 − 0,27) = {-np.log(0.73):.3f} ; M11 : {-np.log(0.98):.3f}')
print(f'somme des pertes arrondies = {pertes.round(2).sum():.2f} ; perte moyenne = {pertes.mean():.2f} '
      f'(somme non arrondie : {pertes.sum():.3f})')

# Ajuster d'abord : la perte moyenne en fonction de b (w fixé), comme la courbe de perte du chapitre 3
def perte_b(bb):
    return perte(y, sigmoide(X @ w + bb)).mean()
for bb in (-5, -4, -3):
    print(f'b = {bb} : perte moyenne = {perte_b(bb):.2f}')
bs = np.linspace(-7, -1, 6001)
Lb = [perte_b(bb) for bb in bs]
print(f'fond de la courbe : b ≈ {bs[int(np.argmin(Lb))]:.2f}, perte {min(Lb):.4f} (contre {perte_b(-4):.4f} pour b = −4)')
plt.figure(figsize=(6, 3))
plt.plot(bs, Lb, color=BLEU, linewidth=2.2)
plt.scatter([-4], [perte_b(-4)], marker='D', color=ORANGE, zorder=3)
plt.xlabel('Constante b'); plt.ylabel('Perte moyenne'); plt.grid(alpha=0.4); plt.show()
print('Écart au carré pour le spam noté 0,01 :', round((1 - 0.01) ** 2, 2), '; pour 0,50 :', (1 - 0.5) ** 2)

In [ ]:
# Leçon 5.5 — comparer des seuils : alertes, fausses alertes, manqués, précision, rappel, F1, coût (k = 3)
lignes = []
for seuil in (0.25, 0.3, 0.5, 0.6, 0.7, 0.96):
    sig = p_exact >= seuil                              # valeurs exactes
    vp, fp = int((sig & est_spam).sum()), int((sig & ~est_spam).sum())
    fn = int((~sig & est_spam).sum())
    precision = None if vp + fp == 0 else vp / (vp + fp)   # colonne des signalés vide → indéfinie
    lignes.append({'seuil': seuil, 'alertes': vp + fp, 'fausses alertes': fp, 'manqués': fn,
                   'précision': affiche(precision), 'rappel': f'{vp / (vp + fn):.2f}',
                   'F1': f'{2 * vp / (2 * vp + fp + fn):.2f}', 'coût (k = 3)': fn + 3 * fp})
print(pd.DataFrame(lignes).to_string(index=False))
print(f'F1 au seuil 0,3 : 2 × 0,8 × 1 / 1,8 = {2 * 0.8 * 1 / 1.8:.2f} ; au seuil 0,6 : 1,5 / 1,75 = {1.5 / 1.75:.2f}')

# Le même calcul avec scikit-learn, s'il est installé : sa convention pour une précision indéfinie est 0
try:
    from sklearn.metrics import precision_score, f1_score
    predit = (p_exact >= 0.96).astype(int)
    print('scikit-learn au seuil 0,96 : précision =', precision_score(y, predit, zero_division=0),
          '(convention zero_division=0) ; F1 =', f1_score(y, predit, zero_division=0))
except ImportError:
    print('scikit-learn absent : ce calcul est facultatif.')

## Exercice 5.1 — Lire un graphique : deux filtres face à la diagonale

*Notions : probabilité estimée et calibration (leçon 5.2).* Lisez d'abord la figure du livre. Filtre A : données D5.2 ; filtre B : données D5.E1.

### a) à d) Vos lectures

*Vos réponses :* …

In [ ]:
# À COMPLÉTER : e) fréquences observées du filtre B (spams / effectif de chaque tranche),
# puis nombre de spams attendu dans la tranche 0,9–1 si B était calibré (p̂ moyenne × effectif)
freq_b = None
attendu_derniere = None

In [ ]:
# Vérification de e)
if freq_b is None or attendu_derniere is None:
    print('pas encore complété. Complétez la cellule précédente.')
else:
    print('fréquences de B :', np.round(freq_b, 2), '→',
          'correct' if np.allclose(freq_b, [0.10, 0.24, 0.43, 0.55, 0.62, 56 / 75]) else 'à revoir')
    n, q = calibration_b['messages'].iloc[-1], calibration_b['proba_moyenne'].iloc[-1]
    et = np.sqrt(n * q * (1 - q))                       # écart-type du nombre de spams (exercice 4.6)
    observe = calibration_b['spams'].iloc[-1]
    print(f'tranche 0,9–1 : attendu {attendu_derniere:.2f} spams, observé {observe}, écart-type {et:.2f}, '
          f'écart = {(attendu_derniere - observe) / et:.0f} écarts-types →',
          'correct' if abs(attendu_derniere - 72.75) < 0.01 else 'à revoir')
    plt.figure(figsize=(4.2, 4.2))
    plt.plot([0, 1], [0, 1], ':', color='gray', label='calibration parfaite')
    plt.plot(calibration['proba_moyenne'], calibration['spams'] / calibration['messages'], 'o-', color=BLEU, label='filtre A')
    plt.plot(calibration_b['proba_moyenne'], freq_b, '^--', color=VERT, label='filtre B')
    plt.xlabel('Probabilité moyenne prédite'); plt.ylabel('Fréquence observée'); plt.legend(); plt.grid(alpha=0.4)
    plt.show()

### e) Votre interprétation

*Votre réponse :* …

## Exercice 5.2 — Trois seuils pour une association

*Notions : règle de décision et matrice de confusion (leçon 5.1), rappel, taux de fausses alertes et précision (équation 5.2), F1 et coût (leçon 5.5).* Dix messages vérifiés par le secrétariat (données D5.E2), trois seuils : 0,3, 0,5 et 0,7. **Les probabilités du tableau sont les valeurs à utiliser telles quelles.** Faites d'abord les questions a) à f) sur papier.

### a) à c) Sur papier

*Vos matrices et vos calculs :* …

### b) La matrice de confusion en Python

In [ ]:
# À COMPLÉTER : les quatre cases de la matrice de confusion pour un seuil donné
def matrice_confusion(scores, est_spam, seuil):
    """scores et est_spam : deux listes de même longueur. Renvoie le dictionnaire
    {'VP': ..., 'FN': ..., 'FP': ..., 'VN': ...}. Règle : signalé si score >= seuil."""
    return None   # À COMPLÉTER

In [ ]:
# Vérification : d'abord sur les douze messages (valeurs EXACTES), puis sur les dix messages de l'exercice
cas = [('leçon, seuil 0,5', list(p_exact), list(est_spam), 0.5, (3, 1, 1, 7)),
       ('leçon, seuil 0,25', list(p_exact), list(est_spam), 0.25, (4, 0, 2, 6)),
       ('exercice, seuil 0,7', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.7, (2, 3, 0, 5)),
       ('exercice, seuil 0,5', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.5, (4, 1, 1, 4)),
       ('exercice, seuil 0,3', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.3, (4, 1, 3, 2))]
for nom, sc, sp, seuil, attendu in cas:
    m = matrice_confusion(sc, sp, seuil)
    if m is None:
        print(f'{nom} : pas encore complété. Complétez la fonction.')
    else:
        obtenu = (m['VP'], m['FN'], m['FP'], m['VN'])
        print(f'{nom} : VP, FN, FP, VN = {obtenu} →', 'correct' if obtenu == attendu else 'à revoir')

### c) et e) Les mesures et le F1

In [ ]:
# À COMPLÉTER : rappel, taux de fausses alertes, précision (équation 5.2) et F1 (équation 5.8).
# Renvoyez None pour une mesure dont le dénominateur est nul (convention du notebook).
def mesures(m):
    rappel = None                 # À COMPLÉTER : dénominateur = ligne des spams
    taux_fausses_alertes = None   # À COMPLÉTER : dénominateur = ligne des messages légitimes
    precision = None              # À COMPLÉTER : dénominateur = colonne des signalés
    f1 = None                     # À COMPLÉTER : 2 VP / (2 VP + FP + FN)
    return rappel, taux_fausses_alertes, precision, f1

In [ ]:
# Vérification de c) et e)
scores_e2, spams_e2 = list(dix['score']), list(dix['etiquette'] == 'spam')
if matrice_confusion(scores_e2, spams_e2, 0.5) is None:
    print('Complétez d\'abord matrice_confusion.')
elif mesures(matrice_confusion(scores_e2, spams_e2, 0.5))[0] is None:
    print('Mesures : pas encore complété. Complétez la fonction mesures.')
else:
    attendus = {0.7: (0.4, 0.0, 1.0), 0.5: (0.8, 0.2, 0.8), 0.3: (0.8, 0.6, 4 / 7)}
    for seuil, att in attendus.items():
        r, f, p, _ = mesures(matrice_confusion(scores_e2, spams_e2, seuil))
        print(f'seuil {seuil} : rappel = {affiche(r)}, fausses alertes = {affiche(f)}, précision = {affiche(p)} →',
              'correct' if np.allclose((r, f, p), att) else 'à revoir')
    r, f, p, f1 = mesures(matrice_confusion(scores_e2, spams_e2, 0.95))
    print(f'seuil 0.95 : aucune alerte ; rappel = {affiche(r)}, précision = {affiche(p)}, F1 = {affiche(f1)} →',
          'correct' if r == 0 and p is None and f1 == 0 else 'à revoir')
    f1s = [mesures(matrice_confusion(scores_e2, spams_e2, s))[3] for s in (0.7, 0.5, 0.3)]
    print('F1 aux seuils 0,7 / 0,5 / 0,3 :', [round(v, 2) for v in f1s], '→',
          'correct' if np.allclose(f1s, [4 / 7, 0.8, 8 / 12]) else 'à revoir')

### f) Le coût des erreurs

In [ ]:
# À COMPLÉTER : coût total = spams manqués + k × fausses alertes (équation 5.9)
def cout(m, k):
    return None   # À COMPLÉTER

In [ ]:
# Vérification de f)
if matrice_confusion(scores_e2, spams_e2, 0.5) is None or cout({'FN': 0, 'FP': 0}, 3) is None:
    print('Coût : pas encore complété. Complétez les fonctions.')
else:
    for k, attendu in ((3, [3, 4, 10]), (1, [3, 2, 4])):
        couts = [cout(matrice_confusion(scores_e2, spams_e2, s), k) for s in (0.7, 0.5, 0.3)]
        meilleur = (0.7, 0.5, 0.3)[couts.index(min(couts))]
        print(f'k = {k} : coûts aux seuils 0,7 / 0,5 / 0,3 = {couts}, meilleur seuil {meilleur} →',
              'correct' if couts == attendu else 'à revoir')

### g) Toutes les valeurs du seuil

In [ ]:
# Rappel, précision et F1 pour les seuils 0, 0,01, …, 1 (précision indéfinie : pas de point)
if matrice_confusion(scores_e2, spams_e2, 0.5) is None or mesures(matrice_confusion(scores_e2, spams_e2, 0.5))[0] is None:
    print('Complétez matrice_confusion et mesures pour tracer les courbes.')
else:
    seuils = np.round(np.arange(0, 1.001, 0.01), 2)
    R, F, P, F1 = zip(*[mesures(matrice_confusion(scores_e2, spams_e2, s)) for s in seuils])
    P = [np.nan if p is None else p for p in P]
    plt.figure(figsize=(6, 3.2))
    plt.step(seuils, R, where='post', color=SPAM, linewidth=2, label='rappel (ligne des spams)')
    plt.step(seuils, P, where='post', color=SIGNALE, linewidth=2, linestyle='--', label='précision (colonne des signalés)')
    plt.step(seuils, F1, where='post', color='black', linewidth=1.5, linestyle=':', label='F1')
    for s in (0.3, 0.5, 0.7):
        plt.axvline(s, color=SEUIL, linewidth=1, alpha=0.6)
    plt.xlabel('Seuil'); plt.ylim(-0.05, 1.05); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### d) à f) Vos interprétations

*Votre réponse d) :* …

*Votre F1 en e) :* …

*Votre choix en f) :* …

## Exercice 5.3 — Interpréter : mêmes décisions, pertes différentes

*Notions : perte logarithmique (leçons 5.3 et 5.4), règle de décision (équation 5.1).* Données D5.E3 : ces probabilités sont les valeurs à utiliser.

### a) à d) Sur papier

*Vos réponses :* …

In [ ]:
# À COMPLÉTER : perte logarithmique moyenne (équations 5.6 et 5.7)
def perte_moyenne(etiquettes, probas):
    """etiquettes : liste de 1 (spam) et 0 (légitime) ; probas : probabilités estimées d'être un spam."""
    return None   # À COMPLÉTER ; np.log est le logarithme naturel

In [ ]:
# Vérification
y4 = list((quatre['etiquette'] == 'spam').astype(int))
if perte_moyenne(y4, quatre['p_filtre_A']) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    p_c = [0.99, 0.01, 0.10, 0.01]
    for nom, probas, att in (('A', quatre['p_filtre_A'], 0.3670), ('B', quatre['p_filtre_B'], 0.0577), ('C', p_c, 1.1827)):
        v = perte_moyenne(y4, probas)
        print(f'filtre {nom} : perte moyenne = {v:.4f} →', 'correct' if abs(v - att) < 0.0001 else 'à revoir')
    print('contrôle avec les douze messages de la leçon 5.4 :', round(perte_moyenne(list(y), list(p_exact)), 2))

## Exercice 5.4 — Le coût décide du seuil

*Notions : coût d'un seuil (équation 5.9), F1 (équation 5.8).* On reprend les douze messages, avec leurs probabilités **exactes** `p_exact` : c'est à elles que s'applique la règle.

### a) Votre prédiction

*Si une fausse alerte coûte de plus en plus cher, le meilleur seuil…* …

In [ ]:
# À COMPLÉTER : coût d'un seuil pour les douze messages (valeurs exactes), k = prix d'une fausse alerte
def cout_total(seuil, k):
    return None   # À COMPLÉTER : spams manqués + k × fausses alertes ; utilisez p_exact et est_spam

In [ ]:
# b) à e) Les meilleurs seuils pour chaque k, puis les seuils du F1
seuils = np.round(np.arange(0, 1.001, 0.01), 2)
if cout_total(0.5, 3) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    attendus = {0.1: (0.27, 0.37), 1: (0.27, 0.62), 3: (0.51, 0.62), 10: (0.51, 0.62)}
    plt.figure(figsize=(6, 3.2))
    for k, att in attendus.items():
        c = np.array([cout_total(s, k) for s in seuils])
        meilleurs = seuils[np.isclose(c, c.min())]
        print(f'k = {k:>4} : coût minimal {c.min():.1f}, atteint pour les seuils de {meilleurs.min():.2f} à {meilleurs.max():.2f} →',
              'correct' if (meilleurs.min(), meilleurs.max()) == att else 'à revoir')
        plt.step(seuils, c / c.max(), where='post', label=f'k = {k}')
    plt.xlabel('Seuil'); plt.ylabel('Coût (divisé par son maximum)'); plt.legend(fontsize=8); plt.grid(alpha=0.4)
    plt.show()
f1 = []
for s in seuils:
    sig = p_exact >= s
    vp, fp, fn = (sig & est_spam).sum(), (sig & ~est_spam).sum(), (~sig & est_spam).sum()
    f1.append(2 * vp / (2 * vp + fp + fn))
f1 = np.array(f1)
best = seuils[np.isclose(f1, f1.max())]
print(f'F1 maximal {f1.max():.2f}, pour les seuils de {best.min():.2f} à {best.max():.2f}')
print('M6 : valeur exacte', round(float(p_exact[5]), 4), '→ signalé au seuil 0,27 ?', bool(p_exact[5] >= 0.27))

### c) à e) Vos interprétations

*Vos réponses :* …

## Exercice 5.5 — Prédire, puis vérifier : un arrondi au bord du seuil

*Notions : règle de décision (équation 5.1), sigmoïde (équation 5.4), F1 (équation 5.8).* Même poids, mais une constante b = −4,01.

### a) à c) Vos prédictions et calculs

*Votre prédiction :* …

In [ ]:
# À COMPLÉTER : la probabilité exacte de M3 avec b = −4,01, et sa décision au seuil 0,5
p_m3 = None            # un nombre (utilisez sigmoide)
signale_m3 = None      # True ou False

In [ ]:
# Vérification, puis d) décisions sur les valeurs exactes et sur les valeurs arrondies
if p_m3 is None or signale_m3 is None:
    print('pas encore complété. Complétez la cellule précédente.')
else:
    print(f'M3 : p̂ = {p_m3:.4f} (affiché {p_m3:.2f}), signalé : {signale_m3} →',
          'correct' if abs(p_m3 - 0.4975) < 0.0001 and signale_m3 is False else 'à revoir')
p_coll = sigmoide(X @ w - 4.01)
for seuil in (0.5, 0.25):
    exact, arrondi = p_coll >= seuil, p_coll.round(2) >= seuil
    diff = list(douze['message'][exact != arrondi])
    print(f'seuil {seuil} : décisions différentes entre valeur exacte et arrondie : {diff if diff else "aucune"}')
sig = p_coll >= 0.5
vp, fp, fn = int((sig & est_spam).sum()), int((sig & ~est_spam).sum()), int((~sig & est_spam).sum())
print(f'collègue, seuil 0,5 : VP = {vp}, FN = {fn}, FP = {fp} ; précision = {vp / (vp + fp):.2f}, '
      f'rappel = {vp / (vp + fn):.2f}, F1 = {2 * vp / (2 * vp + fp + fn):.2f}')

## Exercice 5.6 — Défi : descendre la perte logarithmique

*Notions : perte moyenne (équation 5.7), dérivées partielles et gradient (équation 3.6, leçon 3.4), sigmoïde (équation 5.4).*

### a) à c) Sur papier

*Vos démonstrations et calculs :* …

In [ ]:
# À COMPLÉTER : gradient de la perte moyenne des douze messages (question b)
def gradient(w_, b_):
    """Renvoie (dL/dw, dL/db) : dL/dw est un tableau de deux nombres."""
    return None   # À COMPLÉTER : utilisez X, y et sigmoide

In [ ]:
# Vérification, puis d) 1 000 pas de gradient (η = 0,5) depuis w = (1 ; 0,5), b = −4
def perte_moy(w_, b_):
    p_ = sigmoide(X @ w_ + b_)
    return float(np.mean(-(y * np.log(p_) + (1 - y) * np.log(1 - p_))))

if gradient(w, b) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    gw, gb = gradient(w, b)
    print(f'au départ : dL/db = {gb:.4f}, dL/dw = {np.round(gw, 4)} →',
          'correct' if abs(gb - 0.0062) < 0.0001 and np.allclose(gw, [0.0144, -0.2368], atol=0.0001) else 'à revoir')
    w_, b_ = w.astype(float), float(b)
    for pas in range(1, 1001):
        gw, gb = gradient(w_, b_)
        w_, b_ = w_ - 0.5 * gw, b_ - 0.5 * gb
        if pas in (1, 10, 100, 1000):
            print(f'après {pas:>4} pas : perte = {perte_moy(w_, b_):.4f}, w = {np.round(w_, 2)}, b = {b_:.2f}')
    print('perte finale →', 'correct' if abs(perte_moy(w_, b_) - 0.0308) < 0.0005 else 'à revoir')
# e) Des poids qui séparent parfaitement les douze messages : mots suspects ≥ 3 pour les spams, ≤ 2 pour les légitimes
for facteur in (1, 10):
    print(f'w = (0 ; {2 * facteur}), b = {-5 * facteur} : perte moyenne = {perte_moy(np.array([0, 2.0]) * facteur, -5.0 * facteur):.6f}')

### d) et e) Vos interprétations

*Vos réponses :* …